# ONIA · 2.4. K-Nearest Neighbors

Clasificare și regresie, implementări de la zero și scikit-learn. Exemplele sunt independente și pot fi executate pe rând.


## Exemplul 1. Distanta calculată manual

In [ ]:
import math

def euclidiana(a, b):
    suma = 0.0
    for i in range(len(a)):
        suma += (a[i] - b[i]) ** 2
    return math.sqrt(suma)

def manhattan(a, b):
    suma = 0.0
    for i in range(len(a)):
        suma += abs(a[i] - b[i])
    return suma

print(euclidiana([1, 2], [4, 6]))  # 5.0
print(manhattan([1, 2], [4, 6]))  # 7.0


## Exemplul 2. K-NN clasificare, doar Python de bază

In [ ]:
from math import sqrt

def distanta(a, b):
    suma = 0.0
    for i in range(len(a)):
        suma += (a[i] - b[i]) ** 2
    return sqrt(suma)

def knn_clasificare(X, y, nou, k):
    if len(X) != len(y) or not (1 <= k <= len(X)):
        raise ValueError("Dimensiuni sau k invalide")
    vecini = []
    for i in range(len(X)):
        d = distanta(X[i], nou)
        vecini.append((d, str(y[i])))
    vecini.sort(key=lambda pereche: pereche[0])
    voturi = {}
    for d, eticheta in vecini[:k]:
        voturi[eticheta] = voturi.get(eticheta, 0) + 1
    # Maximul voturilor; egalitățile sunt rezolvate alfabetic.
    return sorted(voturi, key=lambda c: (-voturi[c], c))[0]

X = [[2, 2], [4, 2], [3, 4], [6, 2], [3, 6]]
y = ["A", "B", "A", "B", "B"]
print(knn_clasificare(X, y, [3, 2], 3))  # A


## Exemplul 3. Ponderarea cu distanța, inclusiv cazul d=0

In [ ]:
def vot_ponderat(vecini):
    # vecini = [(distanta, eticheta), ...]
    identici = [(d, c) for d, c in vecini if d == 0]
    if identici:
        vecini = identici
        ponderi = [(1.0, c) for _, c in vecini]
    else:
        ponderi = [(1.0 / d, c) for d, c in vecini]
    scor = {}
    for p, c in ponderi:
        scor[c] = scor.get(c, 0.0) + p
    return sorted(scor, key=lambda c: (-scor[c], c))[0]

print(vot_ponderat([(0.1, "A"), (0.5, "B"), (0.6, "B")]))


## Exemplul 4. K-NN regresie, de la zero

In [ ]:
from math import sqrt

def knn_regresie(X, y, nou, k):
    if len(X) != len(y) or not (1 <= k <= len(X)):
        raise ValueError("Dimensiuni sau k invalide")
    vecini = []
    for i in range(len(X)):
        dist = sqrt(sum((a - b) ** 2
                        for a, b in zip(X[i], nou)))
        vecini.append((dist, y[i]))
    vecini.sort(key=lambda p: p[0])
    valori = [raspuns for _, raspuns in vecini[:k]]
    return sum(valori) / len(valori)

X = [[1], [2], [3], [4], [5]]
y = [10, 12, 18, 20, 24]
print(knn_regresie(X, y, [2.4], 3))  # 13.333...


## Exemplul 5. Regresie cu ponderi inverse și caz identic

In [ ]:
def medie_locala(vecini):
    # Lista (distanta, valoare), deja selectată la k vecini.
    identici = [y for d, y in vecini if d == 0]
    if identici:
        return sum(identici) / len(identici)
    sus = sum(y / d for d, y in vecini)
    jos = sum(1.0 / d for d, y in vecini)
    return sus / jos

print(round(medie_locala([(1, 10), (2, 12), (4, 20)]), 3))


## Exemplul 6. Pipeline previne scurgerile de date

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=3)
)
X = [[20, 3000], [22, 3500], [55, 9000], [57, 10000]]
y = [0, 0, 1, 1]
model.fit(X, y)
print(model.predict([[23, 3300]]))


## Exemplul 7. One-hot manual pentru un vocabular cunoscut

In [ ]:
culori = ["rosu", "verde", "albastru"]

def codifica(culoare):
    return [1 if culoare == c else 0 for c in culori]

for culoare in culori:
    print(culoare, codifica(culoare))


## Exemplul 8. Date mixte: ColumnTransformer + K-NN

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = pd.DataFrame({
    "varsta": [15, 16, 17, 18, 19, 20],
    "ore": [2, 4, 7, 8, 9, 11],
    "curs": ["online", "sala", "online", "sala", "online", "sala"]
})
y = [0, 0, 0, 1, 1, 1]

numerice = Pipeline([
    ("lipsa", SimpleImputer(strategy="median")),
    ("scalare", StandardScaler())
])
categorice = Pipeline([
    ("lipsa", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
transformari = ColumnTransformer([
    ("num", numerice, ["varsta", "ore"]),
    ("cat", categorice, ["curs"])
])
model = Pipeline([
    ("preprocesare", transformari),
    ("knn", KNeighborsClassifier(n_neighbors=3))
])
model.fit(X, y)
exemplu = pd.DataFrame({"varsta": [17], "ore": [10], "curs": ["online"]})
print(model.predict(exemplu))


## Exemplul 9. Iris: clasificare cu train/test și metrice

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5, weights="distance")
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("Matrice confuzie:\n", confusion_matrix(y_test, pred))
print("Primele probabilități:", model.predict_proba(X_test[:2]))


## Exemplul 10. Diabetes: regresie cu pipeline

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=7, weights="distance")
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("MSE:", round(mean_squared_error(y_test, pred), 2))
print("R²:", round(r2_score(y_test, pred), 3))


## Exemplul 11. GridSearchCV caută k, p și votul ponderat

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

X, y = load_iris(return_X_y=True)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])
parametri = {
    "knn__n_neighbors": [1, 3, 5, 7, 11],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cautare = GridSearchCV(model, parametri, cv=cv, scoring="accuracy")
cautare.fit(X, y)
print(cautare.best_params_)
print(round(cautare.best_score_, 3))
